# Hands-On Machine Learning with PyTorch (Aurelion Geron)

## 1) Building Neural Networks with PyTorch

### 4. Building an Image Classifier with PyTorch

In [1]:
import torch 
import torchvision 
import torchvision.transforms.v2 as T

from torch.utils.data import TensorDataset, DataLoader 

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, val_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
                          #pin_memory=True, num_workers=2, prefetch_factor=1,
                          #persistent_workers=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

X_sample, y_sample = train_data[0]

print(f"X_sample is {X_sample}.\n") 
print(f"X_sample has shape {X_sample.shape} and data type {X_sample.dtype}.\n")
print(f"The classes are {train_and_valid_data.classes}.\n")
print(f"X_sample has class {train_and_valid_data.classes[y_sample]}.")

X_sample is Image([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.1529, 0.2353, 0.2000, 0.2235, 0.1961,
         0.2157, 0.2078, 0.1961, 0.1922, 0.1647, 0.1725, 0.1804, 0.1843, 0.2353,
         0.0431],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0078, 0.0000, 0.5216, 0.6196, 0.5922, 0.7529, 0.7725,
         0.8118, 0.7882, 0.8549, 0.8549, 0.

In [2]:
import torch.nn as nn
import torchmetrics

from HelperMethods import HelperML

# Let's build an image classification MLP with 2 hidden layer
class ImageClassifier(nn.Module):

    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes))

    def forward(self, X):
        return self.mlp(X)

torch.manual_seed(42)

n_inputs = 28*28
n_hidden1 = 100
n_hidden2 = 100
n_classes = 10
n_epochs = 10

model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes)


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device="mps"
else:
    device="cpu"

model = model.to(device)

x_entropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0)

helperML = HelperML(device)

history = helperML.train(model, optimizer, x_entropy, accuracy, train_loader, val_loader, n_epochs)

/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/10,
train_loss: 1.1039,
train_metric: 0.6099,
valid_metric: 0.7418.

Epoch 2/10,
train_loss: 0.5987,
train_metric: 0.7897,
valid_metric: 0.7934.

Epoch 3/10,
train_loss: 0.5153,
train_metric: 0.8206,
valid_metric: 0.8190.

Epoch 4/10,
train_loss: 0.4765,
train_metric: 0.8319,
valid_metric: 0.8298.

Epoch 5/10,
train_loss: 0.4528,
train_metric: 0.8418,
valid_metric: 0.8388.

Epoch 6/10,
train_loss: 0.4343,
train_metric: 0.8473,
valid_metric: 0.8420.

Epoch 7/10,
train_loss: 0.4184,
train_metric: 0.8541,
valid_metric: 0.8434.

Epoch 8/10,
train_loss: 0.4073,
train_metric: 0.8565,
valid_metric: 0.8492.

Epoch 9/10,
train_loss: 0.3963,
train_metric: 0.8595,
valid_metric: 0.8520.

Epoch 10/10,
train_loss: 0.3872,
train_metric: 0.8629,
valid_metric: 0.8576.



In [3]:
# Evaluate on test set 
test_accuracy = helperML.evaluate(model, accuracy, test_loader)
print(f"Accuracy of the classifier on test set is {test_accuracy}.")

Accuracy of the classifier on test set is 0.8518000245094299.


#### Finetuning with Optuna (not executed yet)

In [ ]:
import optuna 

# Let's tune the learning rate and the number of hidden units
def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 100)
    
    model = ImageClassifier(n_inputs, n_hidden, n_hidden, n_classes)
    model = model.to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    history = helperML.train(model, optimizer, x_entropy, accuracy, train_loader, val_loader, n_epochs)

    val_acc = helperML.evaluate(model, accuracy, val_loader)
    
    return val_acc

torch.manual_seed(42)

sampler = optuna.samplers.TPESampler(seed=42)       # Tree-structured Parzen Estimator (TPE) algorithm to learn from
                                                    # past results to better select promising hyperparameters
#pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0, interval_steps=1)  # we can also add pruning, but then 
# we have to adjust the objective method making it loop over epochs and setting n_epochs=1 in each call

study = optuna.create_study(direction="maximize", sampler=sampler) #pruner=pruner
study.optimize(objective, n_trials=3)

[I 2026-09-30 17:40:23,049] A new study created in memory with name: no-name-1c65236c-983c-4c8e-81bc-f420e43b0121
[W 2026-09-30 17:41:08,616] Trial 0 failed with parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_27279/2539823238.py", line 13, in objective
    history = helperML.train(model, optimizer, x_entropy, accuracy, train_loader, val_loader, n_epochs)
  File "/home/kobi/Documents/Documents/LaTeX/LaTexFiles/MachineLearning/MLHandsOnPyTorch/LearningCode/BuildingNeuralNetworksWithPyTorch/HelperMethods.py", line 30, in train
    for *X_batch, y_batch in train_loader:
                             ^^^^^^^^^^^^
  File "/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/torch/utils/data/dataloa

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params 
best_metric = study.best_value 

print(f"The best parameters found are {best_params} with a metric value of {best_metric}.")



#### Saving and Loading PyTorch Models

In [4]:
import torch 

# Save the model
save_path = "my_fashion_mnist.pt"
torch.save(model, save_path)        # insecure (s.b.)
    
# Load the model 
loaded_model = torch.load(save_path, weights_only=False)
loaded_model.eval()

idx, batch_new = list(enumerate(test_loader))[0]
X_new, y_new = batch_new
X_new = X_new.to(device)
y_new = y_new.to(device) 

y_pred_logits = loaded_model(X_new)
y_pred = torch.argmax(y_pred_logits, dim=1)
print(f"train_and_valid_data.classes is {train_and_valid_data.classes}")
print(f"y_pred is {y_pred}")
y_pred = train_and_valid_data.classes[list(y_pred)]
print(f"While the true labels of X_new are {y_new}, the predictions are {y_pred}.")

train_and_valid_data.classes is ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
y_pred is tensor([9, 2, 1, 1, 6, 1, 4, 6, 5, 7, 4, 5, 5, 3, 4, 1, 2, 2, 8, 0, 2, 7, 7, 7,
        1, 2, 6, 0, 9, 4, 8, 8], device='cuda:0')


TypeError: list indices must be integers or slices, not list

#### Since torch.save utilizes pickle's serialization format, which is insecure, we should SAVE THE WEIGHTS ONLY 

In [ ]:
# Save weigths only
save_path2 = "my_fashion_mnist_weights_only.pt"
weights = model.state_dict()
print(f"The model's weights are given by {weights}.")
torch.save(weights, save_path2)

# Load model by filling new model with weights
new_model = ImageClassifier(n_inputs, n_hidden1, n_hidden2, n_classes)
loaded_weights = torch.load(save_path2, weights_only=True)
new_model.load_state_dict(loaded_weights)

# now evaluate: new_model.eval(), ...

#### Compiling and Optimizing a PyTorch Model

In [5]:
# Tracing 

torchscript_model_tracing = torch.jit.trace(model, X_new)   # suitable to our model, since our model is not loopy and contains no conditionals,
                                                    # i.e., our model is static 


# Scripting 

torchscript_model_scripting = torch.jit.script(model)      # suitable to dynamic models containing loops and conditionals 


# Optimize the TorchScript
optimized_model = torch.jit.optimize_for_inference(torchscript_model_tracing)   # here we used tracing 

# Save the optimized model
save_path_torchscript = 'my_fashion_mnist_torchscript.pt'
optimized_model.save(save_path_torchscript)

# Load the optimized model 
loaded_torchscript_model = torch.jit.load(save_path_torchscript)

/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/torch/jit/_trace.py:1000: FutureWarning: `torch.jit.trace` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/torch/jit/_trace.py:1139: FutureWarning: `torch.jit.trace_method` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/torch/jit/_freeze.py:230: FutureWarning: `torch.jit.optimize_for_inference` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
/home/kobi/miniforge3/envs/zhang/lib/python3.14/site-packages/torch/jit

#### We can do even better and easier in JIT Compilation and Optimization

In [ ]:
# New (2023 March) JIT compilation and optimization
compiled_model = torch.compile(model)
print(f"compiled_model is {compiled_model}")

# Save compiled/optimized model 
save_path_compiled = "my_fashion_mnist_compiled.pt"
torch.save(compiled_model, save_path_compiled)

compiled_model is OptimizedModule(
  (_orig_mod): ImageClassifier(
    (mlp): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=784, out_features=100, bias=True)
      (2): ReLU()
      (3): Linear(in_features=100, out_features=100, bias=True)
      (4): ReLU()
      (5): Linear(in_features=100, out_features=10, bias=True)
    )
  )
)
